# Chapter 23 – Worked examples

This notebook runs the examples of [Chapter 23](../index.md) live. The code lives in the modules next to it
(`facility_location.py`, `lp_duality.py`, `either_or.py`); the cells only call them, so a change in a module is a
change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import either_or
import facility_location
import lp_duality
import numpy as np

## Facility location: strong against big-M

Three facilities, four customers. Both formulations have the same integer solutions; their LP relaxations differ.
The strong formulation ($x_{ij} \le y_i$) relaxes to the integer optimum, the big-M one
($\sum_j x_{ij} \le |J|\,y_i$) to a fractional $y$ and a lower bound that branch and bound has to close.

In [ ]:
for strong in (True, False):
    name = 'strong' if strong else 'big-M '
    milp, opened = facility_location.solve(strong, relax=False)
    lp, fractional = facility_location.solve(strong, relax=True)
    print(f'{name}: MILP {milp:.1f} open {opened}; LP relaxation {lp:.1f} with y = {fractional}')

## The Lab LP and its dual values

$\max 3x_1 + 5x_2$ subject to $x_1 \le 4$, $2x_2 \le 12$, $3x_1 + 2x_2 \le 18$, $x \ge 0$: five vertices, the
optimum at (2, 6). The shadow prices (0, 1.5, 1) satisfy $b^\top y = z$ and complementary slackness.

In [ ]:
for x, z in sorted(lp_duality.vertices(), key=lambda t: t[1]):
    print(f'vertex {np.round(x, 2)} -> z = {z:.1f}')
z, x, duals = lp_duality.solve_lab_lp()
print(f'optimum {z:.1f} at {x}; shadow prices {duals}; b . y = {lp_duality.B @ duals:.1f}')

## Why a large M weakens the relaxation

Either $x_1 + x_2 \le 4$ or $3x_1 + x_2 \le 6$ on the box $[0, 10]^2$. With the smallest valid big-Ms the LP
relaxation already equals the integer optimum of $\max x_1 + x_2$; with the box bound it is 13.4, and with ten
times that it is the whole box, 20.

In [ ]:
m_tight = either_or.smallest_valid_m()
print('smallest valid (M1, M2):', m_tight, ' box bound:', either_or.M_BOX)
print('integer optimum:', round(either_or.solve(*m_tight, relax=False)[0], 2))
for m1, m2 in (m_tight, either_or.M_BOX, (10 * either_or.M_BOX[0], 10 * either_or.M_BOX[1])):
    print(f'(M1, M2) = ({m1:.0f}, {m2:.0f}): LP relaxation bound {either_or.solve(m1, m2, relax=True)[0]:.2f}')